# Regularização

O material anterior terminou com três otimizadores empatados na validação apesar de perdas de treino bem diferentes. A distância entre o que a rede acerta no treino e o que acerta em dados novos é o **overfitting**, e é o problema deste material. Uma rede com mais parâmetros do que exemplos consegue decorar o conjunto de treino inteiro, ruído incluído, sem que isso a torne melhor em exemplos que nunca viu.

**Regularização** é qualquer mudança no treinamento que abre mão de um pouco de ajuste ao treino em troca de generalização. Este material monta de propósito uma situação de overfitting, com uma rede grande e poucos dados, e vai reduzindo a distância entre treino e validação acrescentando uma técnica de cada vez: a penalização dos pesos, o dropout e o data augmentation.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

In [ ]:
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Dados

O MNIST entra normalizado como nos materiais anteriores. O que muda é o tamanho do conjunto de treino: apenas 500 exemplos, cinquenta por classe, contra 1.000 de validação. É pouco para uma rede de qualquer tamanho, e é isso que provoca o overfitting.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307,), std=(0.3081,)),
])

full_train_set = datasets.MNIST(root="data", train=True, download=True, transform=transform)
full_test_set = datasets.MNIST(root="data", train=False, download=True, transform=transform)

In [ ]:
train_set = Subset(full_train_set, range(500))
validation_set = Subset(full_test_set, range(1000))

train_dataloader = DataLoader(train_set, batch_size=64, shuffle=True)
validation_dataloader = DataLoader(validation_set, batch_size=500, shuffle=False)

print(f"treino: {len(train_set)}, validação: {len(validation_set)}")

## O modelo

A rede é uma MLP com duas camadas ocultas de 512 unidades, maior do que as dos materiais anteriores: são quase 670 mil parâmetros para 500 exemplos, mais de mil parâmetros por exemplo. O argumento `dropout` insere uma `nn.Dropout` depois de cada ativação, e fica em zero até a seção que trata dele.

In [ ]:
class MLP(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        sizes = [28 * 28, 512, 512]
        layers = [nn.Flatten()]   # [batch, 1, 28, 28] -> [batch, 784]

        for in_features, out_features in zip(sizes, sizes[1:]):
            layers.append(nn.Linear(in_features, out_features))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))

        layers.append(nn.Linear(sizes[-1], 10))
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)   # [batch, 10]


print(f"parâmetros: {sum(p.numel() for p in MLP().parameters())}")

Para enxergar o overfitting é preciso comparar treino e validação, então a função `evaluate` devolve a perda e a acurácia de um dataloader qualquer, e a de treino a chama nos dois ao fim de cada época. Ela ganhou dois argumentos opcionais que ficam nos seus valores padrão até as seções que os usam: `dataloader`, para trocar o conjunto de treino, e `l1_lambda`.

In [ ]:
criterion = nn.CrossEntropyLoss()


def evaluate(model, dataloader):
    model.eval()
    running_loss, correct = 0.0, 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            running_loss += criterion(logits, labels).item() * images.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()

    return running_loss / len(dataloader.dataset), correct / len(dataloader.dataset)

In [ ]:
def train(model, optimizer, dataloader=train_dataloader, l1_lambda=0.0, epochs=40):
    history = {"train_loss": [], "val_loss": [], "train_accuracy": [], "val_accuracy": []}

    for epoch in range(epochs):
        model.train()

        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            loss = criterion(model(images), labels)

            if l1_lambda > 0:
                loss = loss + l1_lambda * sum(p.abs().sum() for p in model.parameters())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        train_loss, train_accuracy = evaluate(model, dataloader)
        val_loss, val_accuracy = evaluate(model, validation_dataloader)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_accuracy"].append(val_accuracy)

    return history

Cada treinamento acrescenta o seu histórico ao dicionário `histories`, e `plot_history` desenha todos os que já existem. Nos dois gráficos o treino é a linha cheia e a validação é a tracejada, na mesma cor; é a distância entre as duas que mede o overfitting.

In [ ]:
histories = {}


def plot_history(histories):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for name, history in histories.items():
        line, = ax1.plot(history["train_loss"], label=name)
        ax1.plot(history["val_loss"], linestyle="--", color=line.get_color())
        ax2.plot(history["train_accuracy"], color=line.get_color(), label=name)
        ax2.plot(history["val_accuracy"], linestyle="--", color=line.get_color())

    ax1.set_ylim(0, 1.5)
    ax1.set_xlabel("época")
    ax1.set_ylabel("entropia cruzada")
    ax1.legend()
    ax1.grid(True)

    ax2.set_ylim(0.4, 1.02)
    ax2.set_xlabel("época")
    ax2.set_ylabel("acurácia")
    ax2.legend()
    ax2.grid(True)
    plt.show()

## Overfitting

O primeiro treinamento não tem regularização nenhuma e serve de referência para todos os outros.

In [ ]:
torch.manual_seed(42)
base_model = MLP().to(device)
histories["sem regularização"] = train(base_model, torch.optim.Adam(base_model.parameters(), lr=1e-3))

plot_history(histories)

É o retrato do overfitting. Na sétima época a rede já acerta 100% do treino e a perda de treino vai a zero, porque 670 mil parâmetros decoram 500 imagens sem esforço. A validação conta outra história: a acurácia trava em 85%, e a perda de validação atinge o mínimo na nona época e depois só sobe, de 0.54 para 0.66. A rede continua treinando, mas tudo o que aprende a partir dali é específico das 500 imagens que viu, e piora a confiança das previsões erradas nas outras.

O ponto em que as curvas de validação param de acompanhar as de treino é o início do overfitting. O que se busca com a regularização é reduzir essa distância, e cada técnica faz isso de um jeito: limitando o tamanho dos pesos, adicionando ruído à rede ou adicionando variação aos dados.

## Penalização dos pesos

A forma mais direta de limitar o que a rede pode decorar é penalizar pesos grandes, somando à perda um termo que cresce com eles,

$$
J(\theta) = L(\theta) + \lambda \, \Omega(\theta)
$$

em que $L$ é a entropia cruzada, $\Omega$ é a penalidade e $\lambda$ controla a força da regularização. A penalidade **L2** é a soma dos quadrados dos pesos, e a **L1** é a soma dos valores absolutos,

$$
\Omega_{L2}(\theta) = \sum_i \theta_i^2
\qquad
\Omega_{L1}(\theta) = \sum_i |\theta_i|
$$

O gradiente da L2 é $2\lambda\theta$, então cada passo encolhe todos os pesos por um fator proporcional ao seu valor: os grandes encolhem muito e os pequenos quase nada. Por isso ela também se chama **weight decay**, e no PyTorch entra como o argumento `weight_decay` do otimizador, sem que seja preciso somar nada à perda. O gradiente da L1 é $\lambda \, \text{sign}(\theta)$, o mesmo empurrão para todos os pesos, o que leva muitos deles a exatamente zero e produz redes esparsas. Não há argumento para ela no PyTorch, e a função `train` a soma à perda quando `l1_lambda` é maior que zero.

Com o Adam há um detalhe: somar $2\lambda\theta$ ao gradiente e depois dividi-lo pela média móvel dos quadrados faz a penalidade ser reescalada por parâmetro, e ela deixa de ser um decaimento uniforme. O **AdamW** corrige isso aplicando o decaimento diretamente nos pesos, fora do passo adaptativo, e é o que se usa quando se quer weight decay com Adam. Com `weight_decay=0` os dois são idênticos.

In [ ]:
torch.manual_seed(42)
l2_model = MLP().to(device)
histories["+ L2"] = train(l2_model, torch.optim.AdamW(l2_model.parameters(), lr=1e-3, weight_decay=1.0))

plot_history(histories)

A perda de treino ainda vai a zero e o treino ainda chega a 100%, mas a perda de validação para de subir: em vez de ir de 0.54 a 0.66, fica em 0.55 do início ao fim. A acurácia de validação quase não muda, porque a rede que decora o treino ainda acerta os mesmos exemplos de validação; o que a penalidade reduz é a confiança exagerada nos erros, que é o que a entropia cruzada mede.

O valor de $\lambda$ depende do otimizador. O decaimento por passo do AdamW é `lr * weight_decay`, e com a taxa de $10^{-3}$ e pouco mais de trezentos passos de treino é preciso um `weight_decay` alto como 1.0 para que ele apareça; com o SGD e uma taxa maior, os valores usuais ficam entre $10^{-4}$ e $10^{-2}$.

O efeito de cada penalidade sobre os pesos aparece no histograma da primeira camada. A rede com L1 é treinada só para essa comparação.

In [ ]:
torch.manual_seed(42)
l1_model = MLP().to(device)
train(l1_model, torch.optim.Adam(l1_model.parameters(), lr=1e-3), l1_lambda=1e-4)

plt.figure(figsize=(8, 5))

for name, model in [("sem regularização", base_model), ("L2", l2_model), ("L1", l1_model)]:
    weights = model.layers[1].weight.detach().cpu().flatten()
    plt.hist(weights, bins=100, range=(-0.1, 0.1), histtype="step", label=name)
    print(f"{name}: {(weights.abs() < 1e-3).float().mean():.1%} dos pesos com |w| < 0.001")

plt.yscale("log")
plt.xlabel("peso")
plt.ylabel("contagem")
plt.legend()
plt.grid(True)
plt.show()

A rede sem regularização e a com L2 têm distribuições parecidas, a segunda um pouco mais concentrada em torno de zero, que é o efeito do encolhimento proporcional. A com L1 é outra coisa: 86% dos pesos da primeira camada estão a menos de 0.001 de zero, contra menos de 3% na rede sem regularização. É a esparsidade que a L1 produz, e a razão de ela ser usada quando se quer descobrir quais entradas importam.

## Dropout

Em vez de limitar os pesos, o **dropout** limita o quanto a rede pode confiar em qualquer unidade isolada. A cada passada de treino, cada unidade da camada é zerada com probabilidade $p$, e as que sobram são reescaladas para compensar,

$$
\tilde{h} = \frac{m \odot h}{1 - p}
\qquad
m_i \sim \text{Bernoulli}(1 - p)
$$

em que $h$ é a saída da camada e $m$ é a máscara sorteada. A divisão por $1 - p$ mantém o valor esperado de $\tilde{h}$ igual a $h$, e é o que permite desligar o dropout na avaliação sem ajustar mais nada: a camada vira a identidade. Como a máscara muda a cada lote, a rede não consegue depender de uma unidade específica, porque ela pode não estar lá no próximo passo, e é forçada a espalhar a informação por várias.

No PyTorch a camada é a `nn.Dropout`, e como a batch normalization, ela se comporta de um jeito no `model.train()` e de outro no `model.eval()`.

In [ ]:
dropout = nn.Dropout(p=0.5)
h = torch.ones(2, 8)

dropout.train()
print(dropout(h))

dropout.eval()
print(dropout(h))

No modo de treino metade das unidades foi zerada e a outra metade dobrou de valor, que é a divisão por $1 - p = 0.5$. No modo de avaliação a saída é igual à entrada. O próximo treinamento acrescenta um dropout de $p = 0.3$ nas duas camadas ocultas à rede com L2 da seção anterior.

In [ ]:
torch.manual_seed(42)
dropout_model = MLP(dropout=0.3).to(device)
histories["+ L2 + dropout"] = train(dropout_model, torch.optim.AdamW(dropout_model.parameters(), lr=1e-3, weight_decay=1.0))

plot_history(histories)

A perda de validação desce mais um degrau, de 0.55 para 0.50, e a acurácia de validação sobe um pouco, para 86%. Mas a rede ainda chega a 100% no treino: com mil parâmetros por exemplo, nem o dropout impede que ela decore as 500 imagens, só a obriga a decorá-las de um jeito mais redundante. Dentro do modelo já se mexeu nos pesos e nas unidades, e o que sobra é mexer nos dados.

## Data augmentation

As técnicas anteriores mexem no modelo. O **data augmentation** mexe nos dados: cada imagem passa por uma transformação aleatória antes de entrar na rede, escolhida entre as que não mudam o rótulo. Um dígito levemente girado, deslocado ou redimensionado continua sendo o mesmo dígito, e a rede, que nunca vê a mesma imagem duas vezes, não consegue decorá-la. A escolha das transformações depende do problema: espelhar uma foto de gato produz outra foto de gato, mas girar um 6 em 180 graus produz um 9.

No `torchvision.transforms` as transformações aleatórias entram no mesmo `Compose` que a conversão para tensor e a normalização, e são sorteadas de novo a cada acesso ao dataset, ou seja, a cada época. A `RandomAffine` combina rotação, translação e escala em uma só.

In [ ]:
augment = transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1))

image, label = datasets.MNIST(root="data", train=True)[0]

fig, axes = plt.subplots(1, 7, figsize=(14, 2.5))
axes[0].imshow(image, cmap="gray")
axes[0].set_title("original")

for i, ax in enumerate(axes[1:], start=1):
    ax.imshow(augment(image), cmap="gray")
    ax.set_title(f"versão {i}")

for ax in axes:
    ax.axis("off")
plt.show()

Cada versão é uma imagem que a rede nunca viu, e todas têm o mesmo rótulo. O conjunto de treino aumentado tem os mesmos 500 índices do original, só que com a transformação no pipeline, e o treinamento acrescenta esse conjunto à rede com L2 e dropout.

In [ ]:
augmented_transform = transforms.Compose([augment, transform])
augmented_train_set = Subset(datasets.MNIST(root="data", train=True, transform=augmented_transform), range(500))
augmented_dataloader = DataLoader(augmented_train_set, batch_size=64, shuffle=True)

torch.manual_seed(42)
augmented_model = MLP(dropout=0.3).to(device)
histories["+ L2 + dropout + augmentation"] = train(
    augmented_model, torch.optim.AdamW(augmented_model.parameters(), lr=1e-3, weight_decay=1.0),
    dataloader=augmented_dataloader,
)

plot_history(histories)

É a maior mudança do material. A acurácia de treino, que era 100% nas três redes anteriores, cai para 92%, e a de validação sobe de 86% para 90%: as duas curvas quase se encontram. A perda de validação cai de 0.50 para 0.32 e ainda está descendo na quadragésima época, sem sinal de subida. As curvas de perda ficam invertidas, com a de treino acima da de validação, porque a rede é avaliada em imagens transformadas, mais difíceis do que as originais, e porque ela nunca consegue decorá-las, já que a cada época elas são outras.

A sequência resume o material: a penalização dos pesos e o dropout contêm o overfitting sem eliminá-lo, porque a rede continua com capacidade de sobra para 500 exemplos, e o augmentation ataca a causa, dando à rede mais variação do que ela consegue decorar. Nenhuma dessas técnicas substitui mais dados: a mesma rede treinada nos 60 mil exemplos do MNIST passa dos 98% sem regularização nenhuma. O que elas fazem é extrair mais generalização dos dados que existem, e são o que se usa quando não há como obter mais.

## Exercícios

### Exercício 1

Treine a rede sem regularização com 2.000 e depois com 10.000 exemplos de treino, mantendo o resto igual. Em que época a perda de validação começa a subir em cada caso, e o que acontece com a distância entre as curvas de treino e validação?

In [ ]:
# train_set = Subset(full_train_set, range(2000))

### Exercício 2

Treine a rede sem L2 nem dropout, só com o augmentation, e compare com a versão que tem os três. Quanto do ganho veio do augmentation sozinho? O que acontece se o dropout sobe para 0.5 com o augmentation ligado?

In [ ]:
# histories["só augmentation"] = train(model, optimizer, dataloader=augmented_dataloader)

### Exercício 3

Troque a `RandomAffine` por uma `transforms.RandomVerticalFlip(p=0.5)` e treine de novo. A acurácia de validação piora? Olhe as imagens de alguns dígitos espelhados e explique o resultado.

In [ ]:
# augment = transforms.RandomVerticalFlip(p=0.5)